In [ ]:
from pathlib import Path

# Локально репозиторий уже открыт. В Colab/Kaggle загружаем его с GitHub.
hosted = Path("/kaggle/working").is_dir() or Path("/content").is_dir()
if hosted:
    work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
    repo_dir = work_dir / "recsys-rl-sequential"
    if repo_dir.exists():
        !git -C {repo_dir} pull --ff-only
    else:
        !git clone https://github.com/daosistt/recsys-rl-sequential {repo_dir}
    %cd {repo_dir}
    %pip install -q -r requirements.txt


Дообучение reward model: GPT-2 + линейная голова из сохранённой Ridge.

Коэффициенты Ridge переносим в `torch.nn.Linear`. Дальше это нейросетевая регрессия по MSE: градиенты обновляют голову и весь GPT-2. Используем AdamW с weight decay; это продолжение обучения, а не повторный вызов sklearn.Ridge.

Подвыборки и временные границы берём из сохранённых экспериментов. Validation MAE определяет лучший чекпоинт, test используется только после выбора модели. Цель — `played_ratio_pct / 100`, без обрезания выше 1. Дополнительных табличных признаков в этом эксперименте нет.

Сравнения Ridge и CatBoost сохранены в `notebooks/reward_comparison.ipynb`.

In [ ]:
from src import config as project_config

project_config.setup_paths()
from src.config import *
from src.prepare_data import load_yambda_data, train_val_test_split
from src.reward_model import clear_memory
from src.reward_finetuning import (
    initialize_unfrozen_reward,
    prepare_reward_loaders,
    train_unfrozen_reward,
    evaluate_unfrozen_reward,
)

print(f"Данные: {DATA_DIR}")
print(f"Результаты: {ARTIFACTS_DIR}")
print(f"Устройство: {device}")


In [ ]:
baseline_dir = ARTIFACTS_DIR / "baseline_t4_1000_minfreq5"
ridge_dir = ARTIFACTS_DIR / "reward_linear_minfreq5"
output_dir = ARTIFACTS_DIR / "reward_unfrozen_minfreq5"
params = REWARD_UNFROZEN_PARAMS.copy()
torch.manual_seed(params["seed"])
np.random.seed(params["seed"])

reward_model, reward_vocab, baseline_config, ridge_config = initialize_unfrozen_reward(
    baseline_dir, ridge_dir, device,
)
experiment = {"baseline_run": baseline_dir.name, "ridge_run": ridge_dir.name,
              "baseline_config": baseline_config, "ridge_config": ridge_config}


In [ ]:
reward_df = load_yambda_data(size=baseline_config["training"]["data_size"])
reward_train, reward_val, reward_test = train_val_test_split(reward_df, **baseline_config["split"])
loaders = prepare_reward_loaders(
    reward_train, reward_val, reward_test, reward_vocab,
    reward_model.config.n_positions, ridge_config["sampling"], params["batch_size"],
)
# После подготовки загрузчиков исходные таблицы уже не нужны.
del reward_df, reward_train, reward_val, reward_test
clear_memory()
print({name: len(loader.dataset) for name, loader in loaders.items()})


In [ ]:
# Обновляем веса копии GPT-2 и головы. Исходные артефакты не перезаписываем.
reward_model, history = train_unfrozen_reward(
    reward_model, loaders["train"], loaders["val"], params, output_dir, experiment,
)
history


In [ ]:
# train_unfrozen_reward уже восстановил веса лучшей эпохи по validation MAE.
test_metrics = evaluate_unfrozen_reward(reward_model, loaders["test"])
frozen_metrics = pd.read_csv(ridge_dir / "metrics.csv", index_col=["Split", "Model"])
comparison = pd.DataFrame([
    frozen_metrics.loc[("test", "GPT-2 + Ridge")].to_dict(),
    test_metrics,
], index=["Frozen GPT-2 + Ridge", "Unfrozen GPT-2 + linear"])
comparison.index.name = "Model"
comparison.to_csv(output_dir / "test_metrics.csv")
clear_memory()
comparison
